# Slack versus unbalanced GPU penalties on hard 15-minute instances

This notebook tests two QUBO encodings of the same GPU-capacity inequality on the synthetic A100 instances used in notebook 21. Both formulations use exact 16-node allocation units, the same assignment variables, energy term, decoder, and physical feasibility check.

For slot $t$, let $L_t(x)$ be active node demand and $G$ capacity. The baseline uses binary slack,

$$H_{G}^{\mathrm{slack}}=\lambda_G\sum_t(G-L_t-S_t)^2,$$

while unbalanced penalization uses no auxiliary variables,

$$H_{G}^{\mathrm{UP}}=\sum_t[-\lambda_1(G-L_t)+\lambda_2(G-L_t)^2].$$

Seeds 7601 and 7603 are development instances. Seed 7602 is held out until coefficient selection is complete. Simulated annealing and PCE are heuristic methods, so feasibility and physical cost are evaluated after decoding.

In [2]:
from __future__ import annotations

from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from dwave.samplers import SimulatedAnnealingSampler
import sys
REPO_ROOT = next(
    directory
    for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / 'src').is_dir()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.config import ModelConfig
from src.quantum.benchmarking import evaluate_physical_schedule, make_effective_price_qubo_inputs
from src.quantum.pce import (
    ANSATZ_BUILDERS, FixedAlphaConfig, ProgressiveBinarizationConfig,
    build_statevector_correlation_evaluator, minimum_pce_qubits,
    pauli_labels_for_qubo, run_fixed_alpha_grid, run_progressive_binarization,
)
from src.quantum.qubo_builder import (
    QuboPenaltyWeights, UnbalancedGpuPenaltyWeights, build_scheduling_qubo,
    decode_qubo_sample, summarize_qubo_graph, to_dimod_bqm, validate_decoded_schedule,
)


## 1. Experiment configuration

Annealing is enabled by default. PCE is opt-in because an unbalanced QUBO with roughly 700 variables maps to about 23 order-two PCE qubits, making exact statevector evaluations expensive. The held-out seed is never used to select penalty coefficients.

In [3]:
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
INSTANCE_ROOT = ROOT / 'data/instances/single_cluster_15m/long_duration_focused'
PROFILE = 'long_wide_window'
DEVELOPMENT_SEEDS = (7601, 7603)
TARGET_SEED = 7602
DELTA_T = 0.25

RUN_ANNEALING_CALIBRATION = True
RUN_HELD_OUT_ANNEALING = True
ANNEALING_READS = 500
ANNEALING_SEED = 41
ASSIGNMENT_GRID = (1_000.0, 10_000.0)
SLACK_GPU_GRID = (0.1, 1.0, 10.0)
UP_LINEAR_GRID = (1.0, 10.0, 100.0)
UP_QUADRATIC_GRID = (0.1, 1.0, 10.0)
PEAK_WEIGHT = 0.0

RUN_GREEDY_UP_CALIBRATION = True
GREEDY_MAX_STEPS = 10
GREEDY_INITIAL_ASSIGNMENT = 10_000.0
GREEDY_INITIAL_QUADRATIC = 0.1
GREEDY_INITIAL_RESERVE_FRACTION = 0.10
GREEDY_GROWTH_FACTOR = 2.0
GREEDY_RESERVE_INCREMENT = 0.05

RUN_PCE = False
PCE_ENCODINGS = ('normalized_slack', 'normalized_unbalanced')
PCE_SEEDS = (7601, 7602)
PCE_ALPHA_GRID = (0.5, 2.0, 8.0)
PCE_MAXITER = 60
PCE_MAX_STATEVECTOR_QUBITS = 23
PCE_ANSATZ = 'legacy_chain_ry_rz_cx_v1'
PCE_LAYERS = 1
PCE_RANDOM_SEED = 73


## 2. Load and normalize the instances

The common divisor is derived from every job demand and cluster capacity. Dividing both by this value is exact: multiplying the normalized demands by `nodes_per_unit` reconstructs the original physical nodes. Job power is unchanged because normalization changes resource units, not physical consumption.

In [4]:
def load_instance(seed: int):
    directory = INSTANCE_ROOT / f'{PROFILE}_seed_{seed}'
    return (
        pd.read_csv(directory / 'jobs.csv'),
        pd.read_csv(directory / 'clusters.csv'),
        pd.read_csv(directory / 'energy_inputs_15m.csv'),
    )

def normalize_node_units(jobs: pd.DataFrame, clusters: pd.DataFrame):
    demands = jobs['node_units_required'].astype(int).to_numpy()
    capacities = clusters['node_capacity'].astype(int).to_numpy()
    nodes_per_unit = int(np.gcd.reduce(np.concatenate([demands, capacities])))
    normalized_jobs = jobs.copy()
    normalized_clusters = clusters.copy()
    normalized_jobs['gpu_count_required'] = normalized_jobs['node_units_required'].astype(int) // nodes_per_unit
    normalized_clusters['gpu_count'] = normalized_clusters['node_capacity'].astype(int) // nodes_per_unit
    normalized_clusters['gpu_capacity'] = normalized_clusters['node_capacity'].astype(int) // nodes_per_unit
    recovered = normalized_jobs['gpu_count_required'] * nodes_per_unit
    if not recovered.equals(normalized_jobs['node_units_required'].astype(int)):
        raise AssertionError('node normalization is not lossless')
    return normalized_jobs, normalized_clusters, nodes_per_unit

def qubo_config(clusters: pd.DataFrame, energy: pd.DataFrame) -> ModelConfig:
    return ModelConfig(
        contracted_power=float(clusters['capacity'].sum()) * 10.0,
        renewable_price=1_000_000.0, peak_price=0.0, delta_t=DELTA_T,
        pue=float(energy['pue'].iloc[0]),
    )

instances = {}
for seed in (*DEVELOPMENT_SEEDS, TARGET_SEED):
    raw_jobs, raw_clusters, energy = load_instance(seed)
    jobs, clusters, nodes_per_unit = normalize_node_units(raw_jobs, raw_clusters)
    instances[seed] = {
        'jobs': jobs, 'clusters': clusters, 'energy': energy,
        'raw_jobs': raw_jobs, 'raw_clusters': raw_clusters,
        'nodes_per_unit': nodes_per_unit,
    }

display(pd.DataFrame([
    {
        'seed': seed, 'jobs': len(data['jobs']), 'nodes_per_unit': data['nodes_per_unit'],
        'physical_capacity_nodes': int(data['raw_clusters']['node_capacity'].iloc[0]),
        'solver_capacity_units': int(data['clusters']['gpu_count'].iloc[0]),
        'minimum_job_units': int(data['jobs']['gpu_count_required'].min()),
        'maximum_job_units': int(data['jobs']['gpu_count_required'].max()),
    }
    for seed, data in instances.items()
]))


,seed,jobs,nodes_per_unit,physical_capacity_nodes,solver_capacity_units,minimum_job_units,maximum_job_units
0,7601,42,16,256,16,1,6
1,7603,42,16,256,16,1,6
2,7602,42,16,256,16,1,6


## 3. Build the two QUBO formulations

`normalized_slack` is exact but introduces binary unused-capacity variables. `normalized_unbalanced` introduces no slack variables, but its two coefficients must shape the energy landscape so overloaded schedules are unattractive. Physical feasibility is always checked after decoding.

In [5]:
def build_qubo(data: dict, encoding: str, assignment: float, gpu_a: float, gpu_b: float = 0.0):
    weights = QuboPenaltyWeights(
        assignment=assignment,
        gpu_capacity=gpu_a if encoding == 'normalized_slack' else 0.0,
        peak_smoothing=PEAK_WEIGHT,
    )
    unbalanced = (
        UnbalancedGpuPenaltyWeights(linear=gpu_a, quadratic=gpu_b)
        if encoding == 'normalized_unbalanced' else None
    )
    return build_scheduling_qubo(
        data['jobs'], make_effective_price_qubo_inputs(data['energy']),
        data['clusters'], qubo_config(data['clusters'], data['energy']), weights,
        unbalanced_gpu_weights=unbalanced,
    )

profile_rows = []
for seed in DEVELOPMENT_SEEDS:
    data = instances[seed]
    examples = {
        'normalized_slack': build_qubo(data, 'normalized_slack', 10_000.0, 1.0),
        'normalized_unbalanced': build_qubo(data, 'normalized_unbalanced', 10_000.0, 10.0, 1.0),
    }
    for encoding, qubo in examples.items():
        coefficients = [abs(v) for v in qubo.linear.values()] + [abs(v) for v in qubo.quadratic.values()]
        profile_rows.append({
            'seed': seed, 'encoding': encoding, **summarize_qubo_graph(qubo),
            'pce_qubits_order_2': minimum_pce_qubits(qubo.num_variables, order=2),
            'max_abs_coefficient': max(coefficients, default=0.0),
        })
qubo_profiles = pd.DataFrame(profile_rows)
display(qubo_profiles[[
    'seed', 'encoding', 'assignment_variables', 'slack_variables', 'logical_variables',
    'quadratic_couplers', 'coupling_density', 'pce_qubits_order_2', 'max_abs_coefficient',
]])


,seed,encoding,assignment_variables,slack_variables,logical_variables,quadratic_couplers,coupling_density,pce_qubits_order_2,max_abs_coefficient
0,7601,normalized_slack,699,480,1179,82111,0.118242,29,21080.0
1,7601,normalized_unbalanced,699,0,699,49751,0.203938,23,21080.0
2,7603,normalized_slack,726,480,1206,99212,0.136540,29,21080.0
3,7603,normalized_unbalanced,726,0,726,60652,0.230463,23,21080.0


## 4. Simulated-annealing calibration on development seeds

Every returned sample is decoded. Ranking first rewards feasible samples, then minimizes missing/extra assignments, overloaded slots, physical node excess, and finally QUBO energy. This prevents a low-energy infeasible schedule from being reported as a successful result.

In [6]:
def violation_summary(schedule, jobs, clusters, hours, nodes_per_unit):
    counts = schedule['job_id'].value_counts() if not schedule.empty else pd.Series(dtype=int)
    missing = sum(int(counts.get(job_id, 0)) == 0 for job_id in jobs['job_id'])
    duplicated = sum(int(count) > 1 for count in counts)
    extras = sum(max(0, int(count) - 1) for count in counts)
    violation_slots, max_excess = 0, 0.0
    for cluster in clusters.itertuples(index=False):
        selected = schedule[schedule['assigned_cluster'] == cluster.cluster_id] if not schedule.empty else schedule
        for hour in hours:
            active = selected[(selected['start_hour'] <= hour) & (hour < selected['start_hour'] + selected['duration'])] if not selected.empty else selected
            excess = max(0.0, float(active['gpu_count_required'].sum()) - float(cluster.gpu_count))
            violation_slots += int(excess > 1e-8)
            max_excess = max(max_excess, excess)
    return {
        'missing_jobs': missing, 'duplicated_jobs': duplicated, 'extra_assignments': extras,
        'assignment_errors': missing + extras, 'gpu_violation_slots': violation_slots,
        'max_gpu_excess_nodes': max_excess * nodes_per_unit,
    }

def sample_qubo(seed: int, encoding: str, assignment: float, gpu_a: float, gpu_b: float = 0.0):
    data = instances[seed]
    qubo = build_qubo(data, encoding, assignment, gpu_a, gpu_b)
    started = perf_counter()
    sampleset = SimulatedAnnealingSampler().sample(
        to_dimod_bqm(qubo), num_reads=ANNEALING_READS, seed=ANNEALING_SEED + seed
    )
    runtime = perf_counter() - started
    candidates = []
    hours = qubo.metadata['hours']
    for record in sampleset.data(fields=['sample', 'energy']):
        schedule = decode_qubo_sample(qubo, dict(record.sample))
        validation = validate_decoded_schedule(schedule, data['jobs'], data['clusters'], hours)
        diagnostics = violation_summary(
            schedule, data['jobs'], data['clusters'], hours, data['nodes_per_unit']
        )
        candidates.append({
            'schedule': schedule, 'validation': validation, 'qubo_energy': float(record.energy),
            **diagnostics,
        })
    feasible = [candidate for candidate in candidates if candidate['validation']['feasible']]
    selected = min(feasible or candidates, key=lambda candidate: (
        not candidate['validation']['feasible'], candidate['assignment_errors'],
        candidate['gpu_violation_slots'], candidate['max_gpu_excess_nodes'], candidate['qubo_energy'],
    ))
    return {
        'seed': seed, 'encoding': encoding, 'assignment': assignment,
        'gpu_linear_or_slack': gpu_a, 'gpu_quadratic': gpu_b,
        'feasible_samples': len(feasible), 'selected_feasible': selected['validation']['feasible'],
        'runtime_s': runtime, 'qubo_variables': qubo.num_variables,
        'quadratic_couplers': len(qubo.quadratic),
        **{key: selected[key] for key in [
            'missing_jobs', 'duplicated_jobs', 'extra_assignments', 'assignment_errors',
            'gpu_violation_slots', 'max_gpu_excess_nodes', 'qubo_energy',
        ]},
        'schedule': selected['schedule'], 'qubo': qubo,
    }

def calibration_specs():
    for assignment in ASSIGNMENT_GRID:
        for gpu_weight in SLACK_GPU_GRID:
            yield 'normalized_slack', assignment, gpu_weight, 0.0
        for linear in UP_LINEAR_GRID:
            for quadratic in UP_QUADRATIC_GRID:
                yield 'normalized_unbalanced', assignment, linear, quadratic

annealing_artifacts = []
if RUN_ANNEALING_CALIBRATION:
    for seed in DEVELOPMENT_SEEDS:
        for specification in calibration_specs():
            annealing_artifacts.append(sample_qubo(seed, *specification))
    calibration_results = pd.DataFrame([
        {key: value for key, value in result.items() if key not in {'schedule', 'qubo'}}
        for result in annealing_artifacts
    ])
    calibration_results['strategy'] = 'grid'
    display(calibration_results.sort_values(
        ['feasible_samples', 'assignment_errors', 'gpu_violation_slots', 'qubo_energy'],
        ascending=[False, True, True, True],
    ).head(20))


,seed,encoding,assignment,gpu_linear_or_slack,gpu_quadratic,feasible_samples,selected_feasible,runtime_s,qubo_variables,quadratic_couplers,missing_jobs,duplicated_jobs,extra_assignments,assignment_errors,gpu_violation_slots,max_gpu_excess_nodes,qubo_energy,strategy
4,7601,normalized_unbalanced,1000.0,1.0,1.0,0,False,7.587147,699,49751,0,0,0,0,6,96.0,3875.978241,grid
31,7603,normalized_unbalanced,1000.0,10.0,1.0,0,False,8.422182,726,60652,0,0,0,0,7,80.0,1083.620078,grid
44,7603,normalized_unbalanced,10000.0,10.0,10.0,0,False,8.326163,726,60652,0,0,0,0,7,80.0,21332.625099,grid
20,7601,normalized_unbalanced,10000.0,10.0,10.0,0,False,7.626887,699,49751,0,0,0,0,7,64.0,22425.379471,grid
47,7603,normalized_unbalanced,10000.0,100.0,10.0,0,False,8.206775,726,60652,0,0,0,0,8,64.0,-2308.549039,grid
7,7601,normalized_unbalanced,1000.0,10.0,1.0,0,False,7.505862,699,49751,0,0,0,0,8,192.0,1726.784745,grid
41,7603,normalized_unbalanced,10000.0,1.0,10.0,0,False,8.373426,726,60652,0,0,0,0,8,80.0,20834.235648,grid
17,7601,normalized_unbalanced,10000.0,1.0,10.0,0,False,7.623539,699,49751,0,0,0,0,8,96.0,23838.836226,grid
23,7601,normalized_unbalanced,10000.0,100.0,10.0,0,False,7.563465,699,49751,0,0,0,0,9,64.0,-6176.473235,grid
28,7603,normalized_unbalanced,1000.0,1.0,1.0,0,False,8.293409,726,60652,0,0,0,0,9,64.0,3655.737152,grid


## 4.1 Constraint-aware greedy calibration

A two-dimensional grid treats $\lambda_1$ and $\lambda_2$ as unrelated, although the unbalanced parabola has its preferred feasible margin at

$$h^*=\frac{\lambda_1}{2\lambda_2}.$$

This calibration chooses a reserve margin $h^*$ as a fraction of normalized capacity and derives $\lambda_1=2\lambda_2h^*$. It then reacts to decoded failures: assignment errors increase $\lambda_A$; GPU overload increases the quadratic scale; repeated overload also shifts the preferred margin farther inside the feasible region. Once both development seeds produce feasible samples, the algorithm tries a smaller GPU scale to find the least-distorting feasible coefficient. QUBO energies are never compared across different coefficient settings.

In [7]:
def evaluate_greedy_step(step, assignment, quadratic, reserve_fraction):
    rows = []
    for seed in DEVELOPMENT_SEEDS:
        capacity = float(instances[seed]['clusters']['gpu_count'].iloc[0])
        reserve_units = reserve_fraction * capacity
        linear = 2.0 * quadratic * reserve_units
        result = sample_qubo(
            seed, 'normalized_unbalanced', assignment, linear, quadratic
        )
        physical_cost = np.nan
        if result['selected_feasible']:
            _, physical = evaluate_physical_schedule(
                result['schedule'], instances[seed]['energy'], delta_t=DELTA_T,
                contracted_power_mw=None, peak_price=0.0,
            )
            physical_cost = physical['physical_total_cost']
        rows.append({
            **{key: value for key, value in result.items() if key not in {'schedule', 'qubo'}},
            'strategy': 'constraint_aware_greedy', 'greedy_step': step,
            'reserve_fraction': reserve_fraction, 'reserve_units': reserve_units,
            'physical_total_cost': physical_cost,
        })
    return rows

def run_constraint_aware_greedy():
    assignment = GREEDY_INITIAL_ASSIGNMENT
    quadratic = GREEDY_INITIAL_QUADRATIC
    reserve_fraction = GREEDY_INITIAL_RESERVE_FRACTION
    rows, feasible_region_found, gpu_failure_streak = [], False, 0

    for step in range(1, GREEDY_MAX_STEPS + 1):
        step_rows = evaluate_greedy_step(step, assignment, quadratic, reserve_fraction)
        rows.extend(step_rows)
        all_seeds_feasible = all(row['feasible_samples'] > 0 for row in step_rows)
        assignment_errors = sum(row['assignment_errors'] for row in step_rows)
        gpu_violations = sum(row['gpu_violation_slots'] for row in step_rows)

        if all_seeds_feasible:
            feasible_region_found = True
            quadratic /= np.sqrt(GREEDY_GROWTH_FACTOR)
            gpu_failure_streak = 0
        elif feasible_region_found:
            break
        elif assignment_errors > 0:
            assignment *= GREEDY_GROWTH_FACTOR
        elif gpu_violations > 0:
            quadratic *= GREEDY_GROWTH_FACTOR
            gpu_failure_streak += 1
            if gpu_failure_streak >= 2:
                reserve_fraction = min(0.50, reserve_fraction + GREEDY_RESERVE_INCREMENT)
                gpu_failure_streak = 0
        else:
            quadratic *= GREEDY_GROWTH_FACTOR
    return pd.DataFrame(rows)

greedy_up_results = pd.DataFrame()
if RUN_GREEDY_UP_CALIBRATION:
    greedy_up_results = run_constraint_aware_greedy()
    greedy_step_summary = greedy_up_results.groupby(
        ['greedy_step', 'assignment', 'gpu_linear_or_slack', 'gpu_quadratic',
         'reserve_fraction', 'reserve_units'], as_index=False
    ).agg(
        development_seeds_feasible=('selected_feasible', 'sum'),
        feasible_samples=('feasible_samples', 'sum'),
        assignment_errors=('assignment_errors', 'sum'),
        gpu_violation_slots=('gpu_violation_slots', 'sum'),
        max_gpu_excess_nodes=('max_gpu_excess_nodes', 'max'),
        mean_physical_cost=('physical_total_cost', 'mean'),
    )
    display(greedy_step_summary)


,greedy_step,assignment,gpu_linear_or_slack,gpu_quadratic,reserve_fraction,reserve_units,development_seeds_feasible,feasible_samples,assignment_errors,gpu_violation_slots,max_gpu_excess_nodes,mean_physical_cost
0,1,10000.0,0.32,0.1,0.10,1.6,0,0,0,33,272.0,NaN
1,2,10000.0,0.64,0.2,0.10,1.6,0,0,0,35,288.0,NaN
2,3,10000.0,1.92,0.4,0.15,2.4,0,0,0,36,208.0,NaN
3,4,10000.0,3.84,0.8,0.15,2.4,0,0,0,32,96.0,NaN
4,5,10000.0,10.24,1.6,0.20,3.2,0,0,0,26,288.0,NaN
5,6,10000.0,20.48,3.2,0.20,3.2,0,0,0,27,96.0,NaN
6,7,10000.0,51.20,6.4,0.25,4.0,0,0,0,21,144.0,NaN
7,8,10000.0,102.40,12.8,0.25,4.0,0,0,0,11,96.0,NaN
8,9,10000.0,245.76,25.6,0.30,4.8,0,0,0,7,64.0,NaN
9,10,10000.0,491.52,51.2,0.30,4.8,0,0,0,4,32.0,NaN


## 5. Select coefficients and evaluate the held-out target

Coefficient selection aggregates both development seeds. The target seed is evaluated only after one configuration per encoding has been selected. The comparison emphasizes feasibility; physical energy is reported only for feasible schedules.

In [8]:
def select_development_configuration(results: pd.DataFrame, encoding: str) -> pd.Series:
    subset = results[results['encoding'] == encoding]
    grouped = subset.groupby(
        ['encoding', 'assignment', 'gpu_linear_or_slack', 'gpu_quadratic'], as_index=False
    ).agg(
        development_seeds_feasible=('feasible_samples', lambda values: int((values > 0).sum())),
        feasible_samples=('feasible_samples', 'sum'),
        assignment_errors=('assignment_errors', 'sum'),
        gpu_violation_slots=('gpu_violation_slots', 'sum'),
        max_gpu_excess_nodes=('max_gpu_excess_nodes', 'max'),
        median_runtime_s=('runtime_s', 'median'),
    )
    return grouped.sort_values(
        ['development_seeds_feasible', 'feasible_samples', 'assignment_errors',
         'gpu_violation_slots', 'max_gpu_excess_nodes'],
        ascending=[False, False, True, True, True],
    ).iloc[0]

selected_configurations = {}
target_artifacts = []
if RUN_ANNEALING_CALIBRATION:
    calibration_frames = [calibration_results]
    if not greedy_up_results.empty:
        calibration_frames.append(greedy_up_results)
    all_calibration_results = pd.concat(calibration_frames, ignore_index=True, sort=False)
    for encoding in ('normalized_slack', 'normalized_unbalanced'):
        selected_configurations[encoding] = select_development_configuration(
            all_calibration_results, encoding
        )
    selected_configuration_table = pd.DataFrame(selected_configurations).T.reset_index(drop=True)
    display(selected_configuration_table)

if RUN_HELD_OUT_ANNEALING and selected_configurations:
    for encoding, configuration in selected_configurations.items():
        target_artifacts.append(sample_qubo(
            TARGET_SEED, encoding, float(configuration.assignment),
            float(configuration.gpu_linear_or_slack), float(configuration.gpu_quadratic),
        ))
    target_rows = []
    for result in target_artifacts:
        row = {key: value for key, value in result.items() if key not in {'schedule', 'qubo'}}
        if result['selected_feasible']:
            _, physical = evaluate_physical_schedule(
                result['schedule'], instances[TARGET_SEED]['energy'], delta_t=DELTA_T,
                contracted_power_mw=None, peak_price=0.0,
            )
            row.update(physical)
        target_rows.append(row)
    target_results = pd.DataFrame(target_rows)
    display(target_results)


ValueError: cannot insert encoding, already exists

## 6. PCE and PB-PCE experiment

The selected development coefficients are reused without target tuning. Fixed-alpha PCE tests sensitivity to binarization strength; PB-PCE then performs adaptive continuation. A skipped run is an explicit statevector-resource result, not a failed schedule. No real QPU execution is performed here.

In [ ]:
def pce_experiment(seed: int, encoding: str, configuration: pd.Series) -> dict:
    data = instances[seed]
    qubo = build_qubo(
        data, encoding, float(configuration.assignment),
        float(configuration.gpu_linear_or_slack), float(configuration.gpu_quadratic),
    )
    required_qubits = minimum_pce_qubits(qubo.num_variables, order=2)
    base = {
        'seed': seed, 'role': 'held_out' if seed == TARGET_SEED else 'development',
        'encoding': encoding, 'qubo_variables': qubo.num_variables,
        'quadratic_couplers': len(qubo.quadratic), 'pce_qubits': required_qubits,
    }
    if required_qubits > PCE_MAX_STATEVECTOR_QUBITS:
        return {**base, 'status': 'skipped_statevector_limit'}

    qubits, labels = pauli_labels_for_qubo(qubo, order=2)
    circuit, parameters = ANSATZ_BUILDERS[PCE_ANSATZ]['builder'](qubits, PCE_LAYERS)
    evaluator = build_statevector_correlation_evaluator(circuit, parameters, labels)
    initial_theta = np.random.default_rng(PCE_RANDOM_SEED + seed).uniform(
        -np.pi, np.pi, len(parameters)
    )
    fixed_config = FixedAlphaConfig(maxiter=PCE_MAXITER, optimizer='COBYLA')
    started = perf_counter()
    alpha_runs, selected = run_fixed_alpha_grid(
        PCE_ALPHA_GRID, initial_theta, evaluator, qubo, data['jobs'], data['clusters'],
        qubo.metadata['hours'], fixed_config,
    )
    pb = run_progressive_binarization(
        initial_theta, evaluator, qubo, data['jobs'], data['clusters'], qubo.metadata['hours'],
        ProgressiveBinarizationConfig(
            initial_alpha=min(PCE_ALPHA_GRID), max_stages=4, fixed_alpha=fixed_config
        ),
    )
    candidate = selected.selected_candidate
    pb_candidate = pb.selected_candidate
    return {
        **base, 'status': 'completed', 'runtime_s': perf_counter() - started,
        'fixed_alpha': selected.alpha,
        'fixed_feasible': candidate.validation['feasible'],
        'fixed_qubo_energy': candidate.qubo_energy,
        'pb_stages': len(pb.stages), 'pb_feasible': pb_candidate.validation['feasible'],
        'pb_qubo_energy': pb_candidate.qubo_energy,
        'fixed_violations': '; '.join(candidate.validation['violations']),
        'pb_violations': '; '.join(pb_candidate.validation['violations']),
    }

if RUN_PCE:
    if not selected_configurations:
        raise RuntimeError('run development calibration before PCE coefficient reuse')
    pce_results = pd.DataFrame([
        pce_experiment(seed, encoding, selected_configurations[encoding])
        for seed in PCE_SEEDS for encoding in PCE_ENCODINGS
    ])
    display(pce_results)


## 7. Interpretation rules

- Normalization is exact because all physical demands are reconstructed losslessly.
- Slack encoding is exact at the formulation level but can remain difficult to sample.
- Unbalanced penalization is successful only if decoded schedules satisfy the physical GPU inequality.
- A smaller QUBO or lower QUBO energy is not evidence of a better schedule. Compare feasibility first and physical energy only between feasible schedules.
- Development seeds select coefficients. Seed 7602 measures held-out generalization.
- Simulated annealing and statevector PCE do not demonstrate quantum advantage; they test formulation behavior and resource scaling.